# SSAFY VQA — Qwen3.8-27B 재구성 파이프라인

이 노트북은 기존 Qwen3-VL-8B 학습 코드를 보존하지 않고, Qwen3.8-27B 실험에 필요한 흐름만 남긴 버전입니다.

실행 순서:

1. 새 런타임에서 환경 설치 및 충돌 검사
2. 데이터 준비 및 고정 validation 구성
3. Qwen3.8-27B bf16 zero-shot 평가
4. counting 전용 LoRA 학습
5. validation 검증 후 test 추론 및 제출 파일 생성

기본 학습 범위는 counting 전용입니다. 기존 최고 8B 제출의 non-counting 예측은 유지하고 counting 문항만 Qwen3.8 adapter로 교체해 negative transfer를 막습니다. 전체 유형을 학습하려면 설정 셀의 TRAIN_SCOPE를 "all"로 변경하세요.

기존 체크포인트와 겹치지 않도록 모든 결과는 /content/qwen38_ssafy 아래에 저장합니다.


## 1. 환경 설치

새 Colab 런타임에서 `런타임 > 모두 실행`을 누르면 설치부터 학습까지 순서대로 진행됩니다. transformers/peft를 이미 import한 런타임이라면 먼저 세션을 종료하고 새 런타임에서 시작하세요.


In [ ]:
# 반드시 새 런타임의 첫 셀로 실행합니다. 이미 import된 모듈 위에 재설치하지 않습니다.
import sys
preloaded = [name for name in ("transformers", "peft", "bitsandbytes", "torchao") if name in sys.modules]
assert not preloaded, (
    f"이미 import된 패키지가 있습니다: {preloaded}. 세션을 종료하고 새 런타임에서 '모두 실행'하세요."
)

# Colab 기본 torchao 0.10.0은 peft 0.20.0(>=0.16 요구)과 충돌하며 BF16 LoRA에는 불필요합니다.
!pip -q uninstall -y torchao
!pip -q install --upgrade \
    "transformers==5.15.1" \
    "peft==0.20.0" \
    "accelerate>=1.14.0" \
    "bitsandbytes>=0.50.2" \
    "datasets>=4.0.0" \
    "pillow>=10.4,<12" \
    "pandas>=2.2" \
    "scikit-learn>=1.5" \
    "tqdm>=4.66"

import importlib
import importlib.metadata
import importlib.util
importlib.invalidate_caches()
assert importlib.metadata.version("transformers") == "5.15.1"
assert importlib.metadata.version("peft") == "0.20.0"
assert importlib.util.find_spec("torchao") is None, (
    "torchao 제거가 반영되지 않았습니다. 세션을 종료하고 새 런타임에서 다시 실행하세요."
)
print("환경 설치 및 충돌 검사 완료 — 재시작 없이 다음 셀로 진행합니다.")


> 이 노트북은 새 런타임에서 위 설치 셀을 가장 먼저 실행하는 것을 전제로 합니다. 설치 셀의 검사가 통과하면 재시작하지 않습니다.


## 2. 공통 설정


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import math
import random
import re
import shutil
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."
DEVICE = torch.device("cuda:0")
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("GPU:", torch.cuda.get_device_name(0), f"({GPU_VRAM_GIB:.1f} GiB)")
print("Torch:", torch.__version__, "/ CUDA:", torch.version.cuda)

# A100 80GB 정확도 우선 프로필. OOM일 때만 아래 값을 "8bit"로 바꾸세요.
MODEL_PRECISION = "bf16"  # "bf16", "8bit", "4bit"
OFFICIAL_MODEL_ID = "Qwen/Qwen3.8-27B"
FOUR_BIT_MODEL_ID = "unsloth/Qwen3.8-27B-unsloth-bnb-4bit"
MODEL_ID = FOUR_BIT_MODEL_ID if MODEL_PRECISION == "4bit" else OFFICIAL_MODEL_ID
AMP_DTYPE = torch.bfloat16 if MODEL_PRECISION in {"bf16", "8bit"} else torch.float16
USE_GRAD_SCALER = AMP_DTYPE == torch.float16
if MODEL_PRECISION == "bf16":
    assert GPU_VRAM_GIB >= 75, (
        f"bf16 27B 프로필에는 약 80GB VRAM이 필요합니다. 현재 {GPU_VRAM_GIB:.1f} GiB"
    )

DATA_ROOT = Path("/content")
DRIVE_ARCHIVE = Path("/content/drive/MyDrive/2026-ssafy-15-2-ai.zip")
MOUNT_DRIVE = True
FORCE_EXTRACT = False

# 기존 실험에서 512 이상 해상도는 counting 이득이 없었으므로 정밀도/LoRA에 메모리를 사용합니다.
IMAGE_SIZE = 512
ZERO_SHOT_BATCH_SIZE = 8
INFER_BATCH_SIZE = 8
ENABLE_THINKING = False

TRAIN_SCOPE = "counting"  # "counting" 권장 또는 "all"
RUN_LORA = True
USE_DEV_PSEUDO = True
# 3/5 pseudo-label 1,634개는 시간과 노이즈가 커서 제외하고 4표 합의만 사용합니다.
MIN_DEV_VOTES = 4
PSEUDO_CAP_RATIO_COUNTING = 1.50
PSEUDO_CAP_RATIO_ALL = 0.75
AUX_NUMERIC_TARGET = True

# A100 80GB 시간 절약 프로필: 실제 batch 2, 유효 batch 8, 고품질 데이터 1 epoch.
EPOCHS = 1
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
SAVE_EVERY_UPDATES = 250

RUN_NAME = f"qwen38_27b_{MODEL_PRECISION}_res{IMAGE_SIZE}_{TRAIN_SCOPE}_r{LORA_R}"
OUTPUT_ROOT = Path("/content/qwen38_ssafy") / RUN_NAME
ADAPTER_ROOT = OUTPUT_ROOT / f"{TRAIN_SCOPE}_adapter"
BEST_ADAPTER_DIR = ADAPTER_ROOT / "best"
ZERO_SHOT_VALID_PATH = OUTPUT_ROOT / "zero_shot_valid.csv"
ADAPTER_VALID_PATH = OUTPUT_ROOT / f"{TRAIN_SCOPE}_adapter_valid.csv"
SUBMISSION_PATH = OUTPUT_ROOT / f"submission_qwen38_{TRAIN_SCOPE}.csv"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_SUBMISSION_PATH = Path("/content/submission_res512_epoch2.csv")
NONCOUNT_SOURCE = "auto"  # "auto", "baseline_csv", "qwen38_zero_shot"

LEGACY_BASELINE_NONCOUNT_CORRECT = 318
LEGACY_BASELINE_NONCOUNT_TOTAL = 328

print("MODEL:", MODEL_ID, "/ precision:", MODEL_PRECISION, "/ amp:", AMP_DTYPE)
print("TRAIN_SCOPE:", TRAIN_SCOPE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## 3. 데이터 준비

train.csv, dev.csv, test.csv와 각 이미지 폴더가 /content에 없을 때만 Drive의 압축 파일을 풉니다.


In [ ]:
required = [
    DATA_ROOT / "train.csv", DATA_ROOT / "dev.csv", DATA_ROOT / "test.csv",
    DATA_ROOT / "train", DATA_ROOT / "dev", DATA_ROOT / "test",
]
need_extract = FORCE_EXTRACT or not all(p.exists() for p in required)

if need_extract:
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    assert DRIVE_ARCHIVE.exists(), f"데이터 압축 파일을 찾지 못했습니다: {DRIVE_ARCHIVE}"
    print("압축 해제 중:", DRIVE_ARCHIVE)
    with zipfile.ZipFile(DRIVE_ARCHIVE) as zf:
        zf.extractall(DATA_ROOT)

missing = [str(p) for p in required if not p.exists()]
assert not missing, f"필수 데이터가 없습니다: {missing}"
print("데이터 준비 완료")


## 4. 데이터 로드와 validation 고정

기존 0.91486 모델과 같은 비교 기준을 유지하기 위해 train의 마지막 10%를 legacy validation으로 사용합니다.


In [ ]:
def categorize(question):
    q = str(question)
    if "몇 개" in q or "개수" in q:
        return "counting"
    if "재질" in q or "소재" in q:
        return "material"
    if "색" in q:
        return "color"
    if "종류" in q:
        return "type"
    return "other"

def load_csv(name, required_columns):
    df = pd.read_csv(DATA_ROOT / name)
    missing = set(required_columns) - set(df.columns)
    assert not missing, f"{name} 컬럼 누락: {sorted(missing)}"
    return df

train_df = load_csv(
    "train.csv", ["id", "path", "question", "a", "b", "c", "d", "answer"]
)
dev_df = load_csv(
    "dev.csv",
    ["id", "path", "question", "a", "b", "c", "d",
     "answer1", "answer2", "answer3", "answer4", "answer5"],
)
test_df = load_csv(
    "test.csv", ["id", "path", "question", "a", "b", "c", "d"]
)

for df in (train_df, dev_df, test_df):
    df["category"] = df["question"].map(categorize)

assert train_df["answer"].isin(list("abcd")).all()
assert train_df["id"].is_unique and dev_df["id"].is_unique and test_df["id"].is_unique

split_index = int(len(train_df) * 0.9)
gold_train_df = train_df.iloc[:split_index].copy().reset_index(drop=True)
valid_df = train_df.iloc[split_index:].copy().reset_index(drop=True)

print("train:", len(train_df), "/ dev:", len(dev_df), "/ test:", len(test_df))
print("gold train:", len(gold_train_df), "/ legacy valid:", len(valid_df))
print("\nTest category distribution")
print(test_df["category"].value_counts())
print("\nLegacy valid category distribution")
print(valid_df["category"].value_counts())


## 5. dev 다수결 pseudo-label과 학습 데이터 구성

dev가 counting에 편중되어 있으므로 높은 vote_count를 먼저 선택하고 gold 데이터 대비 비율 상한을 적용합니다.


In [ ]:
ANSWER_COLUMNS = ["answer1", "answer2", "answer3", "answer4", "answer5"]

def majority_vote(row):
    votes = [
        str(row[c]).strip().lower()
        for c in ANSWER_COLUMNS
        if pd.notna(row[c]) and str(row[c]).strip().lower() in set("abcd")
    ]
    if not votes:
        return pd.Series({"answer": None, "vote_count": 0, "vote_margin": 0})

    counts = Counter(votes)
    ordered = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    top_answer, top_count = ordered[0]
    second_count = ordered[1][1] if len(ordered) > 1 else 0
    return pd.Series({
        "answer": top_answer,
        "vote_count": top_count,
        "vote_margin": top_count - second_count,
    })

vote_df = dev_df.apply(majority_vote, axis=1)
dev_pseudo_df = pd.concat([dev_df.drop(columns=ANSWER_COLUMNS), vote_df], axis=1)
dev_pseudo_df = dev_pseudo_df[
    (dev_pseudo_df["vote_count"] >= MIN_DEV_VOTES)
    & dev_pseudo_df["answer"].isin(list("abcd"))
].copy()
dev_pseudo_df["source"] = "dev_pseudo"

gold_train_df["source"] = "gold"
gold_train_df["vote_count"] = 99
gold_train_df["vote_margin"] = 99

if TRAIN_SCOPE == "counting":
    scoped_gold = gold_train_df[gold_train_df["category"] == "counting"].copy()
    scoped_pseudo = dev_pseudo_df[dev_pseudo_df["category"] == "counting"].copy()
    pseudo_ratio = PSEUDO_CAP_RATIO_COUNTING
elif TRAIN_SCOPE == "all":
    scoped_gold = gold_train_df.copy()
    scoped_pseudo = dev_pseudo_df.copy()
    pseudo_ratio = PSEUDO_CAP_RATIO_ALL
else:
    raise ValueError("TRAIN_SCOPE는 'counting' 또는 'all'이어야 합니다.")

selected_pseudo = []
for category, gold_group in scoped_gold.groupby("category"):
    pseudo_group = scoped_pseudo[scoped_pseudo["category"] == category].copy()
    cap = int(math.ceil(len(gold_group) * pseudo_ratio))
    pseudo_group = pseudo_group.sample(frac=1, random_state=SEED)
    pseudo_group = pseudo_group.sort_values(
        ["vote_count", "vote_margin"], ascending=False, kind="stable"
    ).head(cap)
    selected_pseudo.append(pseudo_group)

selected_pseudo_df = (
    pd.concat(selected_pseudo, ignore_index=True)
    if selected_pseudo else scoped_pseudo.iloc[:0].copy()
)

finetune_df = (
    pd.concat([scoped_gold, selected_pseudo_df], ignore_index=True, sort=False)
    if USE_DEV_PSEUDO else scoped_gold.reset_index(drop=True)
)

valid_scope_df = (
    valid_df[valid_df["category"] == "counting"].copy().reset_index(drop=True)
    if TRAIN_SCOPE == "counting"
    else valid_df.copy().reset_index(drop=True)
)

print("gold 학습:", len(scoped_gold))
print("선택 pseudo:", len(selected_pseudo_df))
print("최종 fine-tune:", len(finetune_df))
print("검증 범위:", len(valid_scope_df))
print(finetune_df.groupby(["category", "source"]).size())


## 6. 프롬프트와 공통 유틸리티


In [ ]:
Image.MAX_IMAGE_PIXELS = None

SYSTEM_INSTRUCT = (
    "You are a visual question answering assistant. "
    "Inspect the entire image carefully. For quantity questions, scan it systematically "
    "and count every relevant visible object exactly once. Your response must start with "
    "exactly one lowercase letter: a, b, c, or d."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답은 a, b, c, d 중 하나의 소문자 한 글자로 시작하세요."
    )

def image_path(relative_path):
    path = Path(relative_path)
    return path if path.is_absolute() else DATA_ROOT / path

def parse_choice_count(text):
    text = str(text)
    if "없음" in text:
        return 0
    match = re.search(r"\d+", text)
    return int(match.group()) if match else None

def build_target_text(row):
    letter = str(row["answer"]).strip().lower()
    if not (AUX_NUMERIC_TARGET and row["category"] == "counting"):
        return letter
    count_value = parse_choice_count(row[letter])
    return letter if count_value is None else f"{letter} ({count_value}개)"

def build_messages(row, image, include_answer=False):
    user_text = build_mc_prompt(
        row["question"], row["a"], row["b"], row["c"], row["d"]
    )
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_text},
        ]},
    ]
    if include_answer:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": build_target_text(row)}],
        })
    return messages

def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()

def cuda_memory():
    return {
        "allocated_GB": round(torch.cuda.memory_allocated() / 1e9, 2),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1e9, 2),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


## 7. Qwen3.8-27B 로드

A100 80GB에서는 정확도 손실을 피하기 위해 공식 bf16 체크포인트를 사용합니다. OOM일 때만 설정 셀에서 8bit로 전환합니다.


In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

load_kwargs = {
    "device_map": {"": 0},
    "dtype": AMP_DTYPE,
    "low_cpu_mem_usage": True,
    "trust_remote_code": True,
}
if MODEL_PRECISION == "8bit":
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_skip_modules=[
            "lm_head", "model.visual", ".*\\.visual\\..*",
            "in_proj_a", "in_proj_b", "in_proj_qkv",
        ],
    )

model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

MODEL_INPUT_DEVICE = next(model.parameters()).device
print("입력 device:", MODEL_INPUT_DEVICE)
print("4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("8-bit:", getattr(model, "is_loaded_in_8bit", False))
print("base dtype:", next(p.dtype for p in model.parameters() if p.is_floating_point()))
print("메모리:", cuda_memory())


## 8. a/b/c/d 로그확률 추론

Qwen3.8은 기본 thinking 모델이지만 letter logit 비교에서는 thinking을 끕니다.


In [ ]:
LETTERS = ["a", "b", "c", "d"]

def apply_template(messages, add_generation_prompt):
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=ENABLE_THINKING,
    )

def build_letter_token_ids():
    dummy_messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [{"type": "text", "text": "dummy"}]},
    ]
    prefix = apply_template(dummy_messages, add_generation_prompt=True)
    prefix_ids = processor.tokenizer(prefix, add_special_tokens=False)["input_ids"]

    result = {}
    for letter in LETTERS:
        full_ids = processor.tokenizer(
            prefix + letter, add_special_tokens=False
        )["input_ids"]
        assert full_ids[:len(prefix_ids)] == prefix_ids
        continuation = full_ids[len(prefix_ids):]
        assert continuation, f"{letter} 토큰을 찾지 못했습니다."
        result[letter] = continuation[0]
    return result

LETTER_TOKEN_IDS = build_letter_token_ids()
print("letter token ids:", LETTER_TOKEN_IDS)

def score_dataframe(active_model, dataframe, batch_size=INFER_BATCH_SIZE, desc="Inference"):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    active_model.eval()
    probs_all = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(dataframe), batch_size), desc=desc, unit="batch"):
            chunk = dataframe.iloc[start:start + batch_size]
            images, texts = [], []

            for _, row in chunk.iterrows():
                with Image.open(image_path(row["path"])) as opened:
                    image = opened.convert("RGB")
                messages = build_messages(row, image, include_answer=False)
                texts.append(apply_template(messages, add_generation_prompt=True))
                images.append(image)

            inputs = processor(
                text=texts, images=images, padding=True, return_tensors="pt"
            ).to(MODEL_INPUT_DEVICE)

            with torch.autocast("cuda", dtype=AMP_DTYPE):
                outputs = active_model(
                    **inputs, use_cache=False, logits_to_keep=1
                )

            letter_tensor = torch.tensor(
                [LETTER_TOKEN_IDS[x] for x in LETTERS],
                device=outputs.logits.device,
            )
            logits = outputs.logits[:, -1, :].index_select(-1, letter_tensor)
            probs = torch.softmax(logits.float(), dim=-1)
            probs_all.extend(probs.cpu().tolist())
            del inputs, outputs, logits, probs, images, texts

    processor.tokenizer.padding_side = old_padding_side
    result = dataframe.reset_index(drop=True).copy()
    for i, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = [p[i] for p in probs_all]
    result["pred"] = [LETTERS[int(np.argmax(p))] for p in probs_all]
    result["pred_conf"] = [float(max(p)) for p in probs_all]
    return result

def summarize_predictions(pred_df, title):
    print(f"\n=== {title} ===")
    if "answer" in pred_df:
        pred_df["correct"] = pred_df["pred"] == pred_df["answer"]
        print(
            f"전체 정확도: {pred_df['correct'].mean():.4f} "
            f"({pred_df['correct'].sum()}/{len(pred_df)})"
        )
        print(pred_df.groupby("category")["correct"].agg(["mean", "count"]))
    print("\n예측 분포")
    print(pred_df["pred"].value_counts().sort_index())


## 9. Zero-shot validation

전체 legacy validation을 먼저 평가해 Qwen3.8 전환의 실제 효과를 확인합니다.


In [ ]:
torch.cuda.reset_peak_memory_stats()

zero_shot_valid = score_dataframe(
    model, valid_df, batch_size=ZERO_SHOT_BATCH_SIZE,
    desc="Qwen3.8 zero-shot valid",
)
summarize_predictions(zero_shot_valid, "Qwen3.8-27B zero-shot")
zero_shot_valid.to_csv(ZERO_SHOT_VALID_PATH, index=False)

zero_count = zero_shot_valid[zero_shot_valid["category"] == "counting"]
zero_count_acc = zero_count["correct"].mean()
print(f"\nZero-shot counting: {zero_count_acc:.4f}")
print("기존 8B counting 기준: 0.8389")
print("메모리:", cuda_memory())

if zero_count_acc >= 0.90:
    print("판정: counting specialist로 매우 유망합니다.")
elif zero_count_acc > 0.8389:
    print("판정: 기존 8B보다 개선. counting 전용 LoRA 가치가 있습니다.")
else:
    print("판정: zero-shot은 기존보다 낮음. LoRA 후 반드시 재검증하세요.")


## 10. LoRA Dataset / Collator

counting 모드에서는 정답 글자를 첫 토큰으로 유지하면서 숫자 의미를 보조 supervision으로 추가합니다. 선택지는 epoch마다 재배열합니다.


In [ ]:
def shuffled_choices(row, rng):
    letters = LETTERS.copy()
    values = [str(row[x]) for x in letters]
    gold_index = letters.index(str(row["answer"]).strip().lower())
    order = list(range(4))
    rng.shuffle(order)
    shuffled_values = [values[i] for i in order]
    shuffled_gold = letters[order.index(gold_index)]

    out = row.copy()
    for letter, value in zip(letters, shuffled_values):
        out[letter] = value
    out["answer"] = shuffled_gold
    return out

class VQADataset(Dataset):
    def __init__(self, dataframe, shuffle_choices=False, seed=SEED):
        self.df = dataframe.reset_index(drop=True)
        self.shuffle_choices = shuffle_choices
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index].copy()
        if self.shuffle_choices:
            rng = random.Random(self.seed + self.epoch * len(self.df) + index)
            row = shuffled_choices(row, rng)
        with Image.open(image_path(row["path"])) as opened:
            image = opened.convert("RGB")
        return {
            "prompt_messages": build_messages(row, image, include_answer=False),
            "full_messages": build_messages(row, image, include_answer=True),
            "image": image,
        }

@dataclass
class SupervisedCollator:
    processor: Any

    def __call__(self, batch):
        self.processor.tokenizer.padding_side = "right"
        full_texts, prompt_texts, images = [], [], []

        for sample in batch:
            full_texts.append(
                apply_template(sample["full_messages"], add_generation_prompt=False)
            )
            prompt_texts.append(
                apply_template(sample["prompt_messages"], add_generation_prompt=True)
            )
            images.append(sample["image"])

        encoded = self.processor(
            text=full_texts, images=images, padding=True, return_tensors="pt"
        )
        labels = encoded["input_ids"].clone()
        labels[encoded["attention_mask"] == 0] = -100

        for i, (full_text, prompt_text) in enumerate(zip(full_texts, prompt_texts)):
            # 이미지를 두 번 전처리하지 않고, raw template의 suffix 길이로 target 경계를 계산합니다.
            raw_prompt_ids = self.processor.tokenizer(
                prompt_text, add_special_tokens=False
            )["input_ids"]
            raw_full_ids = self.processor.tokenizer(
                full_text, add_special_tokens=False
            )["input_ids"]
            assert raw_full_ids[:len(raw_prompt_ids)] == raw_prompt_ids, (
                "prompt/full raw template token prefix가 일치하지 않습니다."
            )
            supervised_length = len(raw_full_ids) - len(raw_prompt_ids)
            full_length = int(encoded["attention_mask"][i].sum())
            prompt_length = full_length - supervised_length
            assert 0 < supervised_length < full_length, "assistant target token이 없습니다."
            labels[i, :prompt_length] = -100
            assert (labels[i] != -100).any(), "supervised token이 없는 샘플입니다."

        encoded["labels"] = labels
        return encoded


## 11. LoRA 부착 및 DataLoader

일반 attention, Gated DeltaNet, MLP를 모두 대상으로 하고 vision tower는 제외합니다.


In [ ]:
assert RUN_LORA, "LoRA를 실행하지 않으려면 이 셀부터 건너뛰세요."

# get_peft_model이 모델을 건드리기 전에 Colab 선택 의존성 충돌을 명확히 차단합니다.
import importlib.metadata
import importlib.util
from packaging.version import Version
if importlib.util.find_spec("torchao") is not None:
    torchao_version = Version(importlib.metadata.version("torchao"))
    assert torchao_version >= Version("0.16.0"), (
        f"torchao {torchao_version}는 peft와 충돌합니다. "
        "'!pip uninstall -y torchao' 실행 후 이 셀을 다시 실행하세요."
    )

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

LORA_TARGET_PATTERN = (
    r".*language_model.*\."
    r"(q_proj|k_proj|v_proj|o_proj|"
    r"in_proj_qkv|in_proj_z|out_proj|"
    r"gate_proj|up_proj|down_proj)$"
)

candidate_modules = [
    name for name, module in model.named_modules()
    if re.fullmatch(LORA_TARGET_PATTERN, name)
]
assert candidate_modules, "LoRA 대상 모듈을 찾지 못했습니다."
assert any("in_proj_qkv" in x for x in candidate_modules), "DeltaNet 대상 누락"
assert any("q_proj" in x for x in candidate_modules), "full attention 대상 누락"
assert not any(".visual." in x for x in candidate_modules), "vision tower가 잘못 포함됨"
print("LoRA 대상 모듈 수:", len(candidate_modules))
print("예시:", candidate_modules[:12])

IS_KBIT = bool(
    getattr(model, "is_loaded_in_4bit", False)
    or getattr(model, "is_loaded_in_8bit", False)
)
if IS_KBIT:
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True
    )
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.enable_input_require_grads()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_PATTERN,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
assert trainable_names
assert not any(".visual." in name for name in trainable_names)
assert any("in_proj_qkv" in name for name in trainable_names)
assert any("q_proj" in name for name in trainable_names)

train_dataset = VQADataset(finetune_df, shuffle_choices=True)
valid_dataset = VQADataset(valid_scope_df, shuffle_choices=False)
collator = SupervisedCollator(processor)

train_loader = DataLoader(
    train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
    collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=2,
)
valid_loader = DataLoader(
    valid_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
    collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=2,
)

print("train batches:", len(train_loader), "/ valid batches:", len(valid_loader))
probe = collator([train_dataset[0]])
probe_target_ids = probe["labels"][0][probe["labels"][0] != -100]
assert int(probe_target_ids[0]) in set(LETTER_TOKEN_IDS.values())
print("target masking 확인:", processor.tokenizer.decode(probe_target_ids))
del probe, probe_target_ids
print("메모리:", cuda_memory())


## 12. LoRA 학습

매 epoch의 실제 letter 정확도로 best adapter를 선택합니다.


In [ ]:
# 전체 학습 전에 forward/backward 1회로 OOM과 gradient 연결을 확인
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    torch.cuda.reset_peak_memory_stats()
    model.train()
    trainable_parameters = [p for p in model.parameters() if p.requires_grad]
    smoke_batch = next(iter(train_loader))
    smoke_batch = {k: v.to(MODEL_INPUT_DEVICE) for k, v in smoke_batch.items()}
    try:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            smoke_loss = model(**smoke_batch).loss
        smoke_loss.backward()
        assert all(p.grad is not None for p in trainable_parameters), (
            "일부 LoRA 파라미터에 gradient가 연결되지 않았습니다."
        )
        print("smoke loss:", float(smoke_loss.item()))
        print("smoke test 메모리:", cuda_memory())
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(
            "27B bf16 LoRA OOM: 먼저 TRAIN_BATCH_SIZE=1, GRAD_ACCUM=8로 바꾸세요. "
            "그래도 OOM이면 MODEL_PRECISION='8bit'로 바꾸고 새 런타임에서 실행하세요."
        ) from exc
    finally:
        model.zero_grad(set_to_none=True)
        del smoke_batch
        if 'smoke_loss' in globals():
            del smoke_loss
        clear_cuda()


In [ ]:
from transformers import get_linear_schedule_with_warmup

trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, fused=True
)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_updates = EPOCHS * updates_per_epoch
warmup_steps = max(1, int(total_updates * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_updates
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)

def validation_loss(active_model):
    active_model.eval()
    total_loss, steps = 0.0, 0
    with torch.inference_mode():
        for batch in tqdm(valid_loader, desc="valid loss", leave=False):
            batch = {k: v.to(MODEL_INPUT_DEVICE) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = active_model(**batch).loss
            total_loss += float(loss.item())
            steps += 1
            del batch, loss
    return total_loss / max(steps, 1)

best_accuracy = -1.0
best_loss = float("inf")
history = []
global_update = 0
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    train_dataset.set_epoch(epoch)
    model.train()
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1} train", unit="batch")
    for step, batch in enumerate(progress, start=1):
        batch = {k: v.to(MODEL_INPUT_DEVICE) for k, v in batch.items()}
        remainder = len(train_loader) % GRAD_ACCUM
        current_accum = (
            remainder
            if remainder and step > len(train_loader) - remainder
            else GRAD_ACCUM
        )
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            raw_loss = model(**batch).loss
            scaled_loss = raw_loss / current_accum

        scaler.scale(scaled_loss).backward()
        running_loss += float(raw_loss.item())

        should_update = (step % GRAD_ACCUM == 0) or (step == len(train_loader))
        if should_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_update += 1
            if global_update % SAVE_EVERY_UPDATES == 0:
                step_dir = ADAPTER_ROOT / f"update{global_update}"
                model.save_pretrained(step_dir)
                processor.save_pretrained(step_dir)
                print("중간 adapter 저장:", step_dir)

        progress.set_postfix(loss=f"{raw_loss.item():.4f}")
        del batch, raw_loss, scaled_loss

    epoch_loss = running_loss / max(len(train_loader), 1)
    val_loss = validation_loss(model)
    val_predictions = score_dataframe(
        model, valid_scope_df, batch_size=1,
        desc=f"Epoch {epoch + 1} letter accuracy",
    )
    val_predictions["correct"] = val_predictions["pred"] == val_predictions["answer"]
    val_accuracy = float(val_predictions["correct"].mean())

    epoch_dir = ADAPTER_ROOT / f"epoch{epoch + 1}"
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)

    improved = (
        val_accuracy > best_accuracy
        or (val_accuracy == best_accuracy and val_loss < best_loss)
    )
    if improved:
        best_accuracy = val_accuracy
        best_loss = val_loss
        model.save_pretrained(BEST_ADAPTER_DIR)
        processor.save_pretrained(BEST_ADAPTER_DIR)

    record = {
        "epoch": epoch + 1,
        "train_loss": epoch_loss,
        "valid_loss": val_loss,
        "valid_accuracy": val_accuracy,
        "best": improved,
    }
    history.append(record)
    pd.DataFrame(history).to_csv(ADAPTER_ROOT / "history.csv", index=False)
    print(record)
    clear_cuda()

print(pd.DataFrame(history))
print("best accuracy:", best_accuracy, "/ best loss:", best_loss)
print("best adapter:", BEST_ADAPTER_DIR)
del optimizer, scheduler, scaler
clear_cuda()
print("추론 전 optimizer 메모리 해제:", cuda_memory())


## 13. Best adapter 재로드 및 검증

counting 모드에서는 기존 8B non-counting 결과와 결합한 projected hybrid 정확도도 계산합니다.


In [ ]:
if "best_eval" not in model.peft_config:
    model.load_adapter(
        str(BEST_ADAPTER_DIR), adapter_name="best_eval", is_trainable=False
    )
model.set_adapter("best_eval")
model.eval()

adapter_valid = score_dataframe(
    model, valid_scope_df, batch_size=1, desc="Best adapter valid"
)
summarize_predictions(adapter_valid, f"Best {TRAIN_SCOPE} adapter")
adapter_valid.to_csv(ADAPTER_VALID_PATH, index=False)
adapter_valid["correct"] = adapter_valid["pred"] == adapter_valid["answer"]
scope_correct = int(adapter_valid["correct"].sum())

if TRAIN_SCOPE == "counting":
    projected_correct = LEGACY_BASELINE_NONCOUNT_CORRECT + scope_correct
    projected_total = LEGACY_BASELINE_NONCOUNT_TOTAL + len(adapter_valid)
    print(
        f"\nProjected hybrid legacy valid: "
        f"{projected_correct / projected_total:.4f} "
        f"({projected_correct}/{projected_total})"
    )
    print("기존 전체 baseline: 0.9232 (469/508)")
else:
    print(f"\n전체 adapter valid: {scope_correct / len(adapter_valid):.4f}")

print("메모리:", cuda_memory())


## 14. Test 추론과 제출 생성

counting 모드에서는 counting만 adapter로 예측하고 non-counting은 기존 최고 제출 CSV 또는 Qwen3.8 zero-shot을 사용합니다.


In [ ]:
if TRAIN_SCOPE == "all":
    test_predictions = score_dataframe(
        model, test_df, batch_size=INFER_BATCH_SIZE, desc="Qwen3.8 all test"
    )
    submission = test_predictions[["id", "pred"]].rename(columns={"pred": "answer"})

else:
    test_counting = test_df[
        test_df["category"] == "counting"
    ].copy().reset_index(drop=True)
    counting_predictions = score_dataframe(
        model, test_counting, batch_size=INFER_BATCH_SIZE,
        desc="Qwen3.8 counting test",
    )
    counting_map = dict(zip(counting_predictions["id"], counting_predictions["pred"]))

    resolved_noncount_source = NONCOUNT_SOURCE
    if NONCOUNT_SOURCE == "auto":
        zero_noncount = zero_shot_valid[zero_shot_valid["category"] != "counting"]
        zero_noncount_acc = float(zero_noncount["correct"].mean())
        legacy_noncount_acc = (
            LEGACY_BASELINE_NONCOUNT_CORRECT / LEGACY_BASELINE_NONCOUNT_TOTAL
        )
        resolved_noncount_source = (
            "baseline_csv"
            if BASELINE_SUBMISSION_PATH.exists() and zero_noncount_acc < legacy_noncount_acc
            else "qwen38_zero_shot"
        )
        print(
            f"non-count 자동 선택: {resolved_noncount_source} "
            f"(Qwen3.8 valid={zero_noncount_acc:.4f}, "
            f"legacy valid={legacy_noncount_acc:.4f})"
        )

    if resolved_noncount_source == "baseline_csv":
        assert BASELINE_SUBMISSION_PATH.exists(), (
            f"기존 최고 submission CSV가 필요합니다: {BASELINE_SUBMISSION_PATH}. "
            "경로를 수정하거나 NONCOUNT_SOURCE='qwen38_zero_shot'으로 바꾸세요."
        )
        baseline_submission = pd.read_csv(BASELINE_SUBMISSION_PATH)
        assert set(baseline_submission.columns) >= {"id", "answer"}
        assert baseline_submission["id"].is_unique
        baseline_map = dict(zip(
            baseline_submission["id"],
            baseline_submission["answer"].astype(str).str.lower(),
        ))
        missing_ids = set(test_df["id"]) - set(baseline_map)
        assert not missing_ids, f"baseline submission ID 누락: {len(missing_ids)}개"
        final_answers = [
            counting_map.get(row.id, baseline_map[row.id])
            for row in test_df.itertuples()
        ]

    elif resolved_noncount_source == "qwen38_zero_shot":
        test_noncount = test_df[
            test_df["category"] != "counting"
        ].copy().reset_index(drop=True)
        with model.disable_adapter():
            noncount_predictions = score_dataframe(
                model, test_noncount, batch_size=INFER_BATCH_SIZE,
                desc="Qwen3.8 zero-shot non-counting test",
            )
        noncount_map = dict(zip(
            noncount_predictions["id"], noncount_predictions["pred"]
        ))
        final_answers = [
            counting_map[row.id] if row.category == "counting"
            else noncount_map[row.id]
            for row in test_df.itertuples()
        ]
    else:
        raise ValueError(
            "NONCOUNT_SOURCE는 'auto', 'baseline_csv', 'qwen38_zero_shot' 중 하나여야 합니다."
        )

    submission = pd.DataFrame({"id": test_df["id"], "answer": final_answers})

assert len(submission) == len(test_df)
assert submission["id"].tolist() == test_df["id"].tolist()
assert submission["answer"].isin(LETTERS).all()
assert not submission["answer"].isna().any()

submission.to_csv(SUBMISSION_PATH, index=False)
print("저장:", SUBMISSION_PATH)
print(submission["answer"].value_counts().sort_index())
print("\nCategory x answer")
print(pd.crosstab(test_df["category"], submission["answer"]))


## 15. 결과 백업

adapter, validation 예측, 학습 이력, submission을 Drive에 복사합니다.


In [ ]:
EXPORT_TO_DRIVE = True
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ssafy_qwen38_results")

if EXPORT_TO_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    archive_base = str(OUTPUT_ROOT / f"qwen38_{TRAIN_SCOPE}_adapter")
    archive_path = Path(
        shutil.make_archive(archive_base, "zip", root_dir=BEST_ADAPTER_DIR)
    )

    artifacts = [
        archive_path,
        ZERO_SHOT_VALID_PATH,
        ADAPTER_VALID_PATH,
        ADAPTER_ROOT / "history.csv",
        SUBMISSION_PATH,
    ]
    for artifact in artifacts:
        if artifact.exists():
            target = DRIVE_OUTPUT_DIR / artifact.name
            shutil.copy2(artifact, target)
            print("copied:", target)

print("완료")


## 16. 기존 8B 체크포인트 재추론 및 27B counting 앙상블

이 섹션은 앞의 27B counting 결과를 보존한 뒤 GPU에서 27B를 해제하고, 기존 최고 성능 8B 체크포인트를 불러옵니다.

`EIGHTB_CHECKPOINT_PATH`만 실제 업로드 경로에 맞게 수정하세요. 폴더, ZIP 파일, 또는 체크포인트 폴더 안의 `adapter_model.safetensors` 경로를 지정할 수 있습니다. 기존 최고 모델은 폴더명이 res448이지만 실제 학습 해상도는 512였으므로 `EIGHTB_IMAGE_SIZE=512`를 유지합니다.

검증 정확도가 재현되면 validation/test의 a~d 확률을 저장하고, counting에서는 8B와 27B adapter의 로그확률 가중치를 검증셋으로 선택합니다. non-counting은 기존 8B 예측을 그대로 사용합니다.


In [ ]:
# ===== 사용자 설정: 기존 최고 8B 체크포인트 경로만 확인하세요 =====
EIGHTB_BASE_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
EIGHTB_CHECKPOINT_PATH = Path("/content/qwen3_vl_8b_lora_res448_epoch2")
EIGHTB_IMAGE_SIZE = 512
EIGHTB_INFER_BATCH_SIZE = 8
MIN_EIGHTB_VALID_ACCURACY = 0.915  # 재현 실패 상태로 긴 test 추론을 진행하지 않기 위한 하한

EIGHTB_OUTPUT_ROOT = OUTPUT_ROOT / "ensemble_8b_qwen38"
EIGHTB_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EIGHTB_VALID_PATH = EIGHTB_OUTPUT_ROOT / "qwen3_vl_8b_valid.csv"
EIGHTB_TEST_PATH = EIGHTB_OUTPUT_ROOT / "qwen3_vl_8b_test.csv"
QWEN38_TEST_COUNT_PATH = EIGHTB_OUTPUT_ROOT / "qwen38_adapter_counting_test.csv"
ENSEMBLE_SUBMISSION_PATH = EIGHTB_OUTPUT_ROOT / "submission_8b_qwen38_ensemble.csv"

# 앞에서 계산한 27B 결과를 CPU DataFrame으로 보존합니다.
assert TRAIN_SCOPE == "counting", "이 앙상블 섹션은 TRAIN_SCOPE='counting' 전용입니다."
assert "adapter_valid" in globals(), "13번 셀(Best adapter 검증)을 먼저 실행하세요."
assert "counting_predictions" in globals(), "14번 셀(counting test 추론)을 먼저 실행하세요."

qwen38_valid_count = adapter_valid.copy(deep=True)
qwen38_test_count = counting_predictions.copy(deep=True)
required_prob_columns = {"id", "pred", "prob_a", "prob_b", "prob_c", "prob_d"}
assert required_prob_columns.issubset(qwen38_valid_count.columns)
assert required_prob_columns.issubset(qwen38_test_count.columns)
assert "answer" in qwen38_valid_count.columns
qwen38_test_count.to_csv(QWEN38_TEST_COUNT_PATH, index=False)
print("27B counting test 확률 저장:", QWEN38_TEST_COUNT_PATH)

# ZIP/체크포인트 폴더/폴더 안 safetensors 경로를 모두 처리합니다.
def resolve_8b_checkpoint(raw_path):
    raw_path = Path(raw_path)
    assert raw_path.exists(), f"8B 체크포인트를 찾지 못했습니다: {raw_path}"

    if raw_path.is_file() and raw_path.suffix.lower() == ".zip":
        extract_root = EIGHTB_OUTPUT_ROOT / "extracted_8b_checkpoint"
        extract_root.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(raw_path), str(extract_root))
        raw_path = extract_root
    elif raw_path.is_file():
        raw_path = raw_path.parent

    if (raw_path / "adapter_config.json").exists() or (raw_path / "config.json").exists():
        return raw_path

    candidates = sorted({
        p.parent
        for pattern in ("adapter_config.json", "config.json")
        for p in raw_path.rglob(pattern)
    })
    assert len(candidates) == 1, (
        f"체크포인트 폴더를 하나로 결정할 수 없습니다: {candidates}. "
        "EIGHTB_CHECKPOINT_PATH를 adapter_config.json 또는 config.json이 있는 폴더로 지정하세요."
    )
    return candidates[0]

resolved_8b_checkpoint = resolve_8b_checkpoint(EIGHTB_CHECKPOINT_PATH)
is_8b_lora = (resolved_8b_checkpoint / "adapter_config.json").exists()
print("8B checkpoint:", resolved_8b_checkpoint)
print("checkpoint type:", "LoRA adapter" if is_8b_lora else "full model")

# 8B 로드 전에 27B와 학습 객체를 확실히 해제합니다.
for object_name in [
    "model", "base_model", "optimizer", "scheduler",
    "train_loader", "valid_loader", "collator",
]:
    if object_name in globals():
        del globals()[object_name]
if "processor" in globals():
    del processor
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("27B 해제 후 메모리:", cuda_memory())

from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

processor_source = (
    resolved_8b_checkpoint
    if (resolved_8b_checkpoint / "preprocessor_config.json").exists()
    else EIGHTB_BASE_MODEL_ID
)
processor_8b = AutoProcessor.from_pretrained(
    processor_source,
    min_pixels=EIGHTB_IMAGE_SIZE * EIGHTB_IMAGE_SIZE,
    max_pixels=EIGHTB_IMAGE_SIZE * EIGHTB_IMAGE_SIZE,
    trust_remote_code=True,
)
if processor_8b.tokenizer.pad_token_id is None:
    processor_8b.tokenizer.pad_token = processor_8b.tokenizer.eos_token

eightb_load_kwargs = {
    "device_map": {"": 0},
    "dtype": torch.bfloat16,
    "low_cpu_mem_usage": True,
    "trust_remote_code": True,
}
if is_8b_lora:
    base_model_8b = AutoModelForImageTextToText.from_pretrained(
        EIGHTB_BASE_MODEL_ID, **eightb_load_kwargs
    )
    model_8b = PeftModel.from_pretrained(
        base_model_8b, str(resolved_8b_checkpoint), is_trainable=False
    )
else:
    base_model_8b = None
    model_8b = AutoModelForImageTextToText.from_pretrained(
        str(resolved_8b_checkpoint), **eightb_load_kwargs
    )

model_8b.eval()
EIGHTB_INPUT_DEVICE = next(model_8b.parameters()).device
print("8B 입력 device:", EIGHTB_INPUT_DEVICE)
print("8B 로드 후 메모리:", cuda_memory())


In [ ]:
# 기존 0.91486 제출을 만들 때 사용한 8B 프롬프트를 그대로 재현합니다.
EIGHTB_SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt_8b(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def build_messages_8b(row, image):
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": EIGHTB_SYSTEM_INSTRUCT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": build_mc_prompt_8b(
                        row["question"], row["a"], row["b"], row["c"], row["d"]
                    ),
                },
            ],
        },
    ]

def apply_template_8b(messages):
    return processor_8b.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def build_letter_token_ids_8b():
    dummy_messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": EIGHTB_SYSTEM_INSTRUCT}],
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "dummy"}],
        },
    ]
    prefix = apply_template_8b(dummy_messages)
    prefix_ids = processor_8b.tokenizer(
        prefix, add_special_tokens=False
    )["input_ids"]

    token_ids = {}
    for letter in LETTERS:
        full_ids = processor_8b.tokenizer(
            prefix + letter, add_special_tokens=False
        )["input_ids"]
        assert full_ids[:len(prefix_ids)] == prefix_ids
        continuation = full_ids[len(prefix_ids):]
        assert continuation, f"{letter}의 첫 토큰을 찾지 못했습니다."
        token_ids[letter] = continuation[0]
    return token_ids

EIGHTB_LETTER_TOKEN_IDS = build_letter_token_ids_8b()
print("8B letter token ids:", EIGHTB_LETTER_TOKEN_IDS)

def score_dataframe_8b(dataframe, batch_size=EIGHTB_INFER_BATCH_SIZE, desc="8B inference"):
    old_padding_side = processor_8b.tokenizer.padding_side
    processor_8b.tokenizer.padding_side = "left"
    model_8b.eval()
    probs_all = []

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(dataframe), batch_size), desc=desc, unit="batch"
        ):
            chunk = dataframe.iloc[start:start + batch_size]
            images, texts = [], []

            for _, row in chunk.iterrows():
                with Image.open(image_path(row["path"])) as opened:
                    image = opened.convert("RGB")
                texts.append(apply_template_8b(build_messages_8b(row, image)))
                images.append(image)

            inputs = processor_8b(
                text=texts, images=images, padding=True, return_tensors="pt"
            ).to(EIGHTB_INPUT_DEVICE)

            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = model_8b(
                    **inputs, use_cache=False, logits_to_keep=1
                )

            letter_tensor = torch.tensor(
                [EIGHTB_LETTER_TOKEN_IDS[x] for x in LETTERS],
                device=outputs.logits.device,
            )
            logits = outputs.logits[:, -1, :].index_select(-1, letter_tensor)
            probs = torch.softmax(logits.float(), dim=-1)
            probs_all.extend(probs.cpu().tolist())
            del inputs, outputs, logits, probs, images, texts

    processor_8b.tokenizer.padding_side = old_padding_side
    result = dataframe.reset_index(drop=True).copy()
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = [p[index] for p in probs_all]
    result["pred"] = [LETTERS[int(np.argmax(p))] for p in probs_all]
    result["pred_conf"] = [float(max(p)) for p in probs_all]
    if "answer" in result.columns:
        result["correct"] = result["pred"] == result["answer"]
    return result

# 먼저 validation을 재현합니다. 결과가 지나치게 낮으면 test 5,074개 추론 전에 중단합니다.
eightb_valid = score_dataframe_8b(
    valid_df, batch_size=EIGHTB_INFER_BATCH_SIZE, desc="기존 8B valid"
)
eightb_valid.to_csv(EIGHTB_VALID_PATH, index=False)
summarize_predictions(eightb_valid, "기존 Qwen3-VL-8B checkpoint")

eightb_valid_accuracy = float(eightb_valid["correct"].mean())
eightb_valid_correct = int(eightb_valid["correct"].sum())
print(
    f"기존 기록과 비교: 현재 {eightb_valid_correct}/508 "
    f"vs 기대 469/508 (차이 {eightb_valid_correct - 469:+d})"
)
assert eightb_valid_accuracy >= MIN_EIGHTB_VALID_ACCURACY, (
    f"8B validation 재현 실패: {eightb_valid_accuracy:.4f}. "
    "체크포인트, IMAGE_SIZE=512, 또는 베이스 모델 경로를 확인하세요."
)

eightb_test = score_dataframe_8b(
    test_df, batch_size=EIGHTB_INFER_BATCH_SIZE, desc="기존 8B test"
)
eightb_test.to_csv(EIGHTB_TEST_PATH, index=False)
print("8B valid 확률:", EIGHTB_VALID_PATH)
print("8B test 확률:", EIGHTB_TEST_PATH)
print("8B 추론 완료 메모리:", cuda_memory())


In [ ]:
# ===== 8B + Qwen3.8-27B counting 확률 앙상블 =====
PROB_COLUMNS = [f"prob_{letter}" for letter in LETTERS]
EPS = 1e-8

eightb_count_valid = eightb_valid[
    eightb_valid["category"] == "counting"
].copy()
paired_valid = eightb_count_valid[
    ["id", "answer", "pred", *PROB_COLUMNS]
].merge(
    qwen38_valid_count[["id", "pred", *PROB_COLUMNS]],
    on="id",
    how="inner",
    validate="one_to_one",
    suffixes=("_8b", "_27b"),
)
assert len(paired_valid) == len(eightb_count_valid) == 180
assert paired_valid["answer"].isin(LETTERS).all()

p8_valid = paired_valid[
    [f"prob_{letter}_8b" for letter in LETTERS]
].to_numpy(dtype=np.float64)
p27_valid = paired_valid[
    [f"prob_{letter}_27b" for letter in LETTERS]
].to_numpy(dtype=np.float64)
gold_index = paired_valid["answer"].map({x: i for i, x in enumerate(LETTERS)}).to_numpy()

def log_probability_blend(prob_8b, prob_27b, alpha_8b):
    scores = (
        alpha_8b * np.log(np.clip(prob_8b, EPS, 1.0))
        + (1.0 - alpha_8b) * np.log(np.clip(prob_27b, EPS, 1.0))
    )
    return scores.argmax(axis=1)

alpha_grid = np.round(np.linspace(0.0, 1.0, 101), 2)
weight_rows = []
for alpha_8b in alpha_grid:
    pred_index = log_probability_blend(p8_valid, p27_valid, alpha_8b)
    weight_rows.append({
        "alpha_8b": alpha_8b,
        "alpha_27b": round(1.0 - alpha_8b, 2),
        "count_correct": int((pred_index == gold_index).sum()),
        "count_accuracy": float((pred_index == gold_index).mean()),
    })
weight_search = pd.DataFrame(weight_rows)
best_count_correct = int(weight_search["count_correct"].max())
best_candidates = weight_search[
    weight_search["count_correct"] == best_count_correct
].copy()
best_candidates["distance_from_equal"] = (
    best_candidates["alpha_8b"] - 0.5
).abs()
best_row = best_candidates.sort_values(
    ["distance_from_equal", "alpha_8b"]
).iloc[0]
BEST_ALPHA_8B = float(best_row["alpha_8b"])
BEST_ALPHA_27B = 1.0 - BEST_ALPHA_8B

pred8_correct = paired_valid["pred_8b"].eq(paired_valid["answer"])
pred27_correct = paired_valid["pred_27b"].eq(paired_valid["answer"])
oracle_correct = int((pred8_correct | pred27_correct).sum())
both_wrong = int((~pred8_correct & ~pred27_correct).sum())
ensemble_valid_index = log_probability_blend(
    p8_valid, p27_valid, BEST_ALPHA_8B
)
ensemble_valid_pred = np.array(LETTERS)[ensemble_valid_index]
ensemble_count_correct = int((ensemble_valid_index == gold_index).sum())

noncount_valid_correct = int(
    eightb_valid.loc[eightb_valid["category"] != "counting", "correct"].sum()
)
projected_valid_correct = noncount_valid_correct + ensemble_count_correct
target_correct = math.ceil(0.95 * len(valid_df))

print("\n=== 8B + 27B counting 앙상블 검증 ===")
print(f"8B counting:       {int(pred8_correct.sum())}/180")
print(f"27B counting:      {int(pred27_correct.sum())}/180")
print(f"둘 중 하나 정답:   {oracle_correct}/180 (단순 두 모델 라우팅의 oracle 상한)")
print(f"둘 다 오답:        {both_wrong}/180")
print(
    f"선택 가중치:       8B {BEST_ALPHA_8B:.2f} / "
    f"27B {BEST_ALPHA_27B:.2f}"
)
print(f"앙상블 counting:   {ensemble_count_correct}/180")
print(
    f"Projected 전체:    {projected_valid_correct}/{len(valid_df)} "
    f"= {projected_valid_correct / len(valid_df):.4f}"
)
print(
    f"0.95 기준({target_correct}/{len(valid_df)})까지 "
    f"{max(0, target_correct - projected_valid_correct)}문제"
)

paired_valid["ensemble_pred"] = ensemble_valid_pred
paired_valid["ensemble_correct"] = (
    paired_valid["ensemble_pred"] == paired_valid["answer"]
)
paired_valid.to_csv(
    EIGHTB_OUTPUT_ROOT / "paired_counting_valid_analysis.csv", index=False
)
weight_search.to_csv(
    EIGHTB_OUTPUT_ROOT / "ensemble_weight_search.csv", index=False
)

# validation에서 정한 하나의 가중치를 test counting에 그대로 적용합니다.
eightb_count_test = eightb_test[
    eightb_test["category"] == "counting"
][["id", "pred", *PROB_COLUMNS]].copy()
paired_test = eightb_count_test.merge(
    qwen38_test_count[["id", "pred", *PROB_COLUMNS]],
    on="id",
    how="inner",
    validate="one_to_one",
    suffixes=("_8b", "_27b"),
)
expected_test_count = int((test_df["category"] == "counting").sum())
assert len(paired_test) == expected_test_count

p8_test = paired_test[
    [f"prob_{letter}_8b" for letter in LETTERS]
].to_numpy(dtype=np.float64)
p27_test = paired_test[
    [f"prob_{letter}_27b" for letter in LETTERS]
].to_numpy(dtype=np.float64)
ensemble_test_index = log_probability_blend(
    p8_test, p27_test, BEST_ALPHA_8B
)
paired_test["ensemble_pred"] = np.array(LETTERS)[ensemble_test_index]
counting_ensemble_map = dict(
    zip(paired_test["id"], paired_test["ensemble_pred"])
)

eightb_test_pred_map = dict(zip(eightb_test["id"], eightb_test["pred"]))
final_ensemble_answers = [
    counting_ensemble_map[row.id]
    if row.category == "counting"
    else eightb_test_pred_map[row.id]
    for row in test_df.itertuples()
]
ensemble_submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": final_ensemble_answers,
})
assert len(ensemble_submission) == len(test_df)
assert ensemble_submission["id"].tolist() == test_df["id"].tolist()
assert ensemble_submission["answer"].isin(LETTERS).all()
assert not ensemble_submission["answer"].isna().any()

ensemble_submission.to_csv(ENSEMBLE_SUBMISSION_PATH, index=False)
paired_test.to_csv(
    EIGHTB_OUTPUT_ROOT / "paired_counting_test_predictions.csv", index=False
)
print("\n최종 앙상블 submission:", ENSEMBLE_SUBMISSION_PATH)
print(ensemble_submission["answer"].value_counts().sort_index())

if (
    globals().get("EXPORT_TO_DRIVE", False)
    and globals().get("DRIVE_OUTPUT_DIR") is not None
    and Path(DRIVE_OUTPUT_DIR).exists()
):
    for artifact in [
        EIGHTB_VALID_PATH,
        EIGHTB_TEST_PATH,
        QWEN38_TEST_COUNT_PATH,
        ENSEMBLE_SUBMISSION_PATH,
        EIGHTB_OUTPUT_ROOT / "paired_counting_valid_analysis.csv",
        EIGHTB_OUTPUT_ROOT / "ensemble_weight_search.csv",
    ]:
        shutil.copy2(artifact, Path(DRIVE_OUTPUT_DIR) / artifact.name)
        print("copied:", Path(DRIVE_OUTPUT_DIR) / artifact.name)


## 17. Targeted Set-of-Mark: Grounding DINO Tiny + 기존 8B

외부 API 없이 로컬 오픈 가중치만 사용합니다. Grounding DINO Tiny가 counting 질문의 대상 후보를 찾고, 원본 위에 굵은 색상 박스와 큰 번호를 표시한 뒤 기존 Qwen3-VL-8B 체크포인트로 다시 추론합니다.

SAM은 첫 실험에서 제외합니다. 먼저 가벼운 bounding-box SoM이 validation에서 실제로 도움 되는지 확인하고, 반복 5-fold 교차검증이 기존 8B+27B 앙상블보다 좋아질 때만 test SoM 추론과 새 제출을 생성합니다.

폐쇄망에서는 `SOM_DETECTOR_ID`를 미리 다운로드한 로컬 체크포인트 폴더로 바꾸고 `SOM_LOCAL_FILES_ONLY=True`로 설정하세요.


In [ ]:
# ===== Targeted SoM 설정 =====
from PIL import ImageDraw, ImageFont
import json

SOM_ENABLED = True
SOM_DETECTOR_ID = "IDEA-Research/grounding-dino-tiny"
SOM_LOCAL_FILES_ONLY = False
SOM_BOX_THRESHOLDS = (0.15, 0.20, 0.25, 0.30)
SOM_TEXT_THRESHOLD = 0.20
SOM_NMS_IOU = 0.50
SOM_MAX_BOXES = 24
SOM_DETECT_BATCH_SIZE = 4
SOM_8B_BATCH_SIZE = 8

# validation 안전 게이트
SOM_CV_REPEATS = 20
SOM_WEIGHT_STEP = 0.05
SOM_MIN_GLOBAL_CORRECT = 164
SOM_MIN_CV_GAIN = 1.0
FORCE_SOM_TEST = False

SOM_ROOT = EIGHTB_OUTPUT_ROOT / "som_grounding_dino"
SOM_VALID_IMAGE_ROOT = SOM_ROOT / "valid_marked"
SOM_TEST_IMAGE_ROOT = SOM_ROOT / "test_marked"
SOM_VALID_DETECTION_PATH = SOM_ROOT / "som_valid_detections.csv"
SOM_TEST_DETECTION_PATH = SOM_ROOT / "som_test_detections.csv"
SOM_CV_PATH = SOM_ROOT / "som_cv_summary.csv"
SOM_SUBMISSION_PATH = SOM_ROOT / "submission_8b_qwen38_som.csv"
SOM_ROOT.mkdir(parents=True, exist_ok=True)

# 런타임 변수만 사라진 경우에는 저장된 확률 CSV에서 자동 복구합니다.
som_dataframe_recovery = {
    "eightb_valid": EIGHTB_VALID_PATH,
    "eightb_test": EIGHTB_TEST_PATH,
    "qwen38_valid_count": ADAPTER_VALID_PATH,
    "qwen38_test_count": QWEN38_TEST_COUNT_PATH,
}
for dataframe_name, dataframe_path in som_dataframe_recovery.items():
    if dataframe_name not in globals() and Path(dataframe_path).exists():
        globals()[dataframe_name] = pd.read_csv(dataframe_path)
        print(f"recovered {dataframe_name}: {dataframe_path}")

required_som_objects = [
    "model_8b", "processor_8b", "eightb_valid", "eightb_test",
    "qwen38_valid_count", "qwen38_test_count",
]
missing_som_objects = [name for name in required_som_objects if name not in globals()]
assert not missing_som_objects, (
    f"16번 8B 앙상블 섹션을 먼저 실행하세요. 누락: {missing_som_objects}"
)

def som_detector_labels(question):
    """한국어 counting 질문을 Grounding DINO용 영문 대상 목록으로 변환합니다."""
    question = str(question)
    labels = []

    # 서로 다른 대상의 개수를 함께 묻는 문제도 있으므로 모든 구체 표현을 수집합니다.
    rules = [
        (("플라스틱 병", "페트병", "PET병"), "plastic bottle"),
        (("유리병",), "glass bottle"),
        (("알루미늄 캔",), "aluminum can"),
        (("스티로폼 상자", "발포 스티로폼 상자"), "styrofoam box"),
        (("스티로폼",), "styrofoam container"),
        (("골판지 상자", "종이상자", "종이 상자", "종이류 상자"), "cardboard box"),
        (("종이팩",), "paper carton"),
        (("종이봉투", "종이 봉투"), "paper bag"),
        (("플라스틱 컵",), "plastic cup"),
        (("플라스틱 용기", "플라스틱 용품"), "plastic container"),
        (("유리 용기",), "glass container"),
        (("비닐봉지", "비닐봉투", "비닐 봉지", "비닐 봉투"), "plastic bag"),
        (("플라스틱 뚜껑",), "plastic bottle cap"),
        (("빨대",), "drinking straw"),
        (("형광등",), "fluorescent light tube"),
        (("튜브형 용기",), "squeeze tube"),
        (("플라스틱 포장지",), "plastic wrapper"),
        (("플라스틱 펜",), "plastic pen"),
        (("음료 용기",), "beverage container"),
    ]
    for korean_terms, english_label in rules:
        if any(term in question for term in korean_terms):
            labels.append(english_label)

    # 구체 표현이 없을 때만 넓은 범주의 fallback을 사용합니다.
    if not labels:
        if "캔" in question:
            labels.append("can")
        if "병" in question:
            labels.append("bottle")
        if "컵" in question:
            labels.append("cup")
        if "상자" in question or "박스" in question:
            labels.append("box")
        if "용기" in question:
            labels.append("container")
        if "봉투" in question or "봉지" in question:
            labels.append("bag")
        if "플라스틱" in question:
            labels.append("plastic object")
        if "종이" in question or "골판지" in question:
            labels.append("paper or cardboard object")
        if "금속" in question:
            labels.append("metal object")

    # 완전히 미지원인 특수 counting 질문은 주요 재활용품 후보를 넓게 표시합니다.
    used_fallback = not labels
    if used_fallback:
        labels = [
            "bottle", "can", "cup", "container",
            "cardboard box", "plastic bag",
        ]

    labels = list(dict.fromkeys(labels))
    return labels, used_fallback

def som_threshold_key(threshold):
    return f"{threshold:.2f}".replace(".", "_")

def som_iou_one_to_many(box, boxes):
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    intersection = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])
    area_b = (
        np.maximum(0.0, boxes[:, 2] - boxes[:, 0])
        * np.maximum(0.0, boxes[:, 3] - boxes[:, 1])
    )
    return intersection / np.maximum(area_a + area_b - intersection, 1e-8)

def som_filter_nms(boxes, scores, labels, threshold):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    labels = list(labels)
    if len(boxes) == 0:
        return boxes, scores, []

    keep_threshold = np.flatnonzero(scores >= threshold)
    boxes = boxes[keep_threshold]
    scores = scores[keep_threshold]
    labels = [labels[i] for i in keep_threshold]
    if len(boxes) == 0:
        return boxes, scores, []

    # 서로 다른 synonym이 같은 객체를 잡는 경우까지 제거하도록 class-agnostic NMS를 사용합니다.
    order = scores.argsort()[::-1]
    keep = []
    while len(order) and len(keep) < SOM_MAX_BOXES:
        current = int(order[0])
        keep.append(current)
        if len(order) == 1:
            break
        remaining = order[1:]
        ious = som_iou_one_to_many(boxes[current], boxes[remaining])
        order = remaining[ious < SOM_NMS_IOU]

    boxes = boxes[keep]
    scores = scores[keep]
    labels = [labels[i] for i in keep]

    # 번호가 이미지 위에서 자연스러운 읽기 순서를 갖도록 위→아래, 왼쪽→오른쪽 정렬합니다.
    spatial_order = np.lexsort((boxes[:, 0], boxes[:, 1]))
    return (
        boxes[spatial_order],
        scores[spatial_order],
        [labels[i] for i in spatial_order],
    )

SOM_PALETTE = [
    "#ff1744", "#00c853", "#2962ff", "#ff9100",
    "#aa00ff", "#00b8d4", "#ffd600", "#f50057",
]

def som_font(size):
    candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf",
    ]
    for font_path in candidates:
        if Path(font_path).exists():
            return ImageFont.truetype(font_path, size=size)
    return ImageFont.load_default()

def draw_som_image(image, boxes):
    marked = image.copy().convert("RGB")
    draw = ImageDraw.Draw(marked)
    width, height = marked.size
    line_width = max(4, round(min(width, height) * 0.007))
    font_size = max(18, round(min(width, height) * 0.040))
    font = som_font(font_size)

    for index, box in enumerate(boxes, start=1):
        x1, y1, x2, y2 = [int(round(value)) for value in box]
        x1 = min(max(x1, 0), width - 1)
        y1 = min(max(y1, 0), height - 1)
        x2 = min(max(x2, x1 + 1), width - 1)
        y2 = min(max(y2, y1 + 1), height - 1)
        color = SOM_PALETTE[(index - 1) % len(SOM_PALETTE)]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=line_width)

        text = str(index)
        try:
            text_box = draw.textbbox(
                (0, 0), text, font=font, stroke_width=1
            )
        except (TypeError, ValueError):
            # PIL 기본 bitmap font는 일부 버전에서 stroke textbbox를 지원하지 않습니다.
            if hasattr(font, "getbbox"):
                text_box = font.getbbox(text)
            else:
                fallback_width, fallback_height = draw.textsize(
                    text, font=font
                )
                text_box = (0, 0, fallback_width, fallback_height)
        text_width = text_box[2] - text_box[0]
        text_height = text_box[3] - text_box[1]
        label_x = min(max(x1, 0), max(0, width - text_width - 10))
        label_y = max(0, y1 - text_height - 10)
        draw.rounded_rectangle(
            (
                label_x,
                label_y,
                label_x + text_width + 10,
                label_y + text_height + 8,
            ),
            radius=4,
            fill=color,
            outline="black",
            width=max(1, line_width // 2),
        )
        draw.text(
            (label_x + 5, label_y + 2),
            text,
            fill="white",
            font=font,
            stroke_width=2,
            stroke_fill="black",
        )
    return marked


In [ ]:
# ===== Grounding DINO Tiny 로드 및 validation 후보 생성 =====
from transformers import AutoModelForZeroShotObjectDetection

def load_som_detector():
    detector_processor = AutoProcessor.from_pretrained(
        SOM_DETECTOR_ID,
        local_files_only=SOM_LOCAL_FILES_ONLY,
    )
    detector_model = AutoModelForZeroShotObjectDetection.from_pretrained(
        SOM_DETECTOR_ID,
        dtype=torch.float32,
        low_cpu_mem_usage=True,
        local_files_only=SOM_LOCAL_FILES_ONLY,
    ).to("cuda")
    detector_model.eval()
    return detector_processor, detector_model

def run_som_detector(dataframe, split_name, image_root):
    image_root = Path(image_root)
    for threshold in SOM_BOX_THRESHOLDS:
        (image_root / som_threshold_key(threshold)).mkdir(
            parents=True, exist_ok=True
        )

    records = []
    minimum_threshold = min(SOM_BOX_THRESHOLDS)

    for start in tqdm(
        range(0, len(dataframe), SOM_DETECT_BATCH_SIZE),
        desc=f"Grounding DINO {split_name}",
        unit="batch",
    ):
        chunk = dataframe.iloc[start:start + SOM_DETECT_BATCH_SIZE]
        images, text_labels, fallback_flags = [], [], []

        for _, row in chunk.iterrows():
            with Image.open(image_path(row["path"])) as opened:
                images.append(opened.convert("RGB"))
            labels, used_fallback = som_detector_labels(row["question"])
            text_labels.append(labels)
            fallback_flags.append(used_fallback)

        detector_inputs = som_detector_processor(
            images=images,
            text=text_labels,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        with torch.inference_mode():
            detector_outputs = som_detector_model(**detector_inputs)

        processed = som_detector_processor.post_process_grounded_object_detection(
            detector_outputs,
            input_ids=detector_inputs["input_ids"],
            threshold=minimum_threshold,
            text_threshold=SOM_TEXT_THRESHOLD,
            target_sizes=[(image.height, image.width) for image in images],
            text_labels=text_labels,
        )

        for local_index, ((_, row), image, labels, used_fallback, result) in enumerate(
            zip(
                chunk.iterrows(), images, text_labels,
                fallback_flags, processed,
            )
        ):
            raw_boxes = result["boxes"].detach().float().cpu().numpy()
            raw_scores = result["scores"].detach().float().cpu().numpy()
            result_labels = result.get("text_labels")
            if result_labels is None:
                result_labels = [str(value) for value in result["labels"]]

            record = {
                "id": row["id"],
                "detector_prompt": json.dumps(labels, ensure_ascii=False),
                "used_fallback": bool(used_fallback),
                "raw_boxes": json.dumps(raw_boxes.tolist()),
                "raw_scores": json.dumps(raw_scores.tolist()),
                "raw_labels": json.dumps(list(result_labels), ensure_ascii=False),
            }

            for threshold in SOM_BOX_THRESHOLDS:
                boxes, scores, kept_labels = som_filter_nms(
                    raw_boxes, raw_scores, result_labels, threshold
                )
                key = som_threshold_key(threshold)
                marked_path = (
                    image_root / key / f"{Path(str(row['id'])).stem}.jpg"
                )
                draw_som_image(image, boxes).save(
                    marked_path, quality=95, subsampling=0
                )
                record[f"som_path_{key}"] = str(marked_path)
                record[f"som_count_{key}"] = int(len(boxes))
                record[f"som_scores_{key}"] = json.dumps(scores.tolist())
                record[f"som_labels_{key}"] = json.dumps(
                    kept_labels, ensure_ascii=False
                )

            records.append(record)

        del detector_inputs, detector_outputs, processed, images
        torch.cuda.empty_cache()

    result_df = pd.DataFrame(records)
    assert len(result_df) == len(dataframe)
    assert result_df["id"].is_unique
    return result_df

if SOM_ENABLED:
    som_detector_processor, som_detector_model = load_som_detector()
    print("Grounding DINO 로드 후 메모리:", cuda_memory())

    counting_valid_df = valid_df[
        valid_df["category"] == "counting"
    ].copy().reset_index(drop=True)
    som_valid_detections = run_som_detector(
        counting_valid_df, "valid", SOM_VALID_IMAGE_ROOT
    )
    som_valid_detections.to_csv(SOM_VALID_DETECTION_PATH, index=False)

    print("SoM valid detections:", SOM_VALID_DETECTION_PATH)
    print("fallback questions:", int(som_valid_detections["used_fallback"].sum()))
    for threshold in SOM_BOX_THRESHOLDS:
        key = som_threshold_key(threshold)
        print(
            f"threshold={threshold:.2f}",
            som_valid_detections[f"som_count_{key}"].describe().round(2).to_dict(),
        )

    preview_key = som_threshold_key(0.25)
    for preview_path in som_valid_detections[f"som_path_{preview_key}"].head(3):
        display(Image.open(preview_path).resize((384, 384)))


In [ ]:
# ===== 마킹된 validation 이미지를 기존 8B로 추론 =====
SOM_SYSTEM_INSTRUCT = (
    "You are a visual counting assistant. The image is the original photograph "
    "with colored numbered detector proposals overlaid on it. Inspect the entire "
    "image, reject irrelevant or duplicate boxes, include visible target objects "
    "missed by boxes when possible, and count each target object exactly once. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt_som(row):
    detector_prompt = row["detector_prompt"]
    return (
        f"{row['question']}\n"
        f"(a) {row['a']}\n(b) {row['b']}\n"
        f"(c) {row['c']}\n(d) {row['d']}\n\n"
        f"Detector candidate labels: {detector_prompt}\n"
        "번호 박스는 후보일 뿐이며 중복 또는 오탐일 수 있습니다. "
        "박스가 놓친 대상도 원본 화면에서 확인하세요. "
        "대상에 해당하는 객체만 정확히 센 뒤 정답을 "
        "a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def build_messages_som(row, image):
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SOM_SYSTEM_INSTRUCT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": build_mc_prompt_som(row)},
            ],
        },
    ]

def apply_template_som(messages):
    return processor_8b.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def build_som_letter_token_ids():
    dummy_messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": SOM_SYSTEM_INSTRUCT}],
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "dummy"}],
        },
    ]
    prefix = apply_template_som(dummy_messages)
    prefix_ids = processor_8b.tokenizer(
        prefix, add_special_tokens=False
    )["input_ids"]
    token_ids = {}
    for letter in LETTERS:
        full_ids = processor_8b.tokenizer(
            prefix + letter, add_special_tokens=False
        )["input_ids"]
        assert full_ids[:len(prefix_ids)] == prefix_ids
        continuation = full_ids[len(prefix_ids):]
        assert continuation
        token_ids[letter] = continuation[0]
    return token_ids

SOM_LETTER_TOKEN_IDS = build_som_letter_token_ids()
print("SoM letter token ids:", SOM_LETTER_TOKEN_IDS)

def score_dataframe_som(dataframe, batch_size=SOM_8B_BATCH_SIZE, desc="SoM 8B"):
    old_padding_side = processor_8b.tokenizer.padding_side
    processor_8b.tokenizer.padding_side = "left"
    model_8b.eval()
    probs_all = []

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(dataframe), batch_size), desc=desc, unit="batch"
        ):
            chunk = dataframe.iloc[start:start + batch_size]
            images, texts = [], []
            for _, row in chunk.iterrows():
                with Image.open(row["som_image_path"]) as opened:
                    image = opened.convert("RGB")
                images.append(image)
                texts.append(apply_template_som(build_messages_som(row, image)))

            inputs = processor_8b(
                text=texts, images=images, padding=True, return_tensors="pt"
            ).to(EIGHTB_INPUT_DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = model_8b(
                    **inputs, use_cache=False, logits_to_keep=1
                )

            letter_tensor = torch.tensor(
                [SOM_LETTER_TOKEN_IDS[letter] for letter in LETTERS],
                device=outputs.logits.device,
            )
            logits = outputs.logits[:, -1, :].index_select(-1, letter_tensor)
            probs = torch.softmax(logits.float(), dim=-1)
            probs_all.extend(probs.cpu().tolist())
            del inputs, outputs, logits, probs, images, texts

    processor_8b.tokenizer.padding_side = old_padding_side
    result = dataframe.reset_index(drop=True).copy()
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = [p[index] for p in probs_all]
    result["pred"] = [LETTERS[int(np.argmax(p))] for p in probs_all]
    result["pred_conf"] = [float(max(p)) for p in probs_all]
    if "answer" in result.columns:
        result["correct"] = result["pred"] == result["answer"]
    return result

som_valid_by_threshold = {}
if SOM_ENABLED:
    valid_detection_base = counting_valid_df.merge(
        som_valid_detections, on="id", how="inner", validate="one_to_one"
    )
    assert len(valid_detection_base) == 180

    for threshold in SOM_BOX_THRESHOLDS:
        key = som_threshold_key(threshold)
        active_valid = valid_detection_base.copy()
        active_valid["som_image_path"] = active_valid[f"som_path_{key}"]
        som_result = score_dataframe_som(
            active_valid,
            batch_size=SOM_8B_BATCH_SIZE,
            desc=f"SoM 8B valid t={threshold:.2f}",
        )
        som_valid_by_threshold[threshold] = som_result
        som_result.to_csv(
            SOM_ROOT / f"som8_valid_t{key}.csv", index=False
        )
        print(
            f"SoM threshold={threshold:.2f}: "
            f"{int(som_result['correct'].sum())}/180 "
            f"= {som_result['correct'].mean():.4f}"
        )


In [ ]:
# ===== 반복 5-fold로 SoM threshold와 앙상블 가중치 검증 =====
SOM_EPS = 1e-10
som_prob_columns = [f"prob_{letter}" for letter in LETTERS]
letter_to_index = {letter: index for index, letter in enumerate(LETTERS)}

base_count_valid = eightb_valid[
    eightb_valid["category"] == "counting"
][["id", "answer", *som_prob_columns]].copy()
adapter_count_valid = qwen38_valid_count[
    ["id", *som_prob_columns]
].copy()

som_pair_base = base_count_valid.merge(
    adapter_count_valid,
    on="id",
    how="inner",
    validate="one_to_one",
    suffixes=("_8b", "_27b"),
)
assert len(som_pair_base) == 180
som_gold = som_pair_base["answer"].map(letter_to_index).to_numpy()
som_p8 = som_pair_base[
    [f"prob_{letter}_8b" for letter in LETTERS]
].to_numpy(dtype=np.float64)
som_p27 = som_pair_base[
    [f"prob_{letter}_27b" for letter in LETTERS]
].to_numpy(dtype=np.float64)

som_probability_by_threshold = {}
for threshold, dataframe in som_valid_by_threshold.items():
    aligned = som_pair_base[["id"]].merge(
        dataframe[["id", *som_prob_columns]],
        on="id",
        how="left",
        validate="one_to_one",
    )
    assert not aligned[som_prob_columns].isna().any().any()
    som_probability_by_threshold[threshold] = aligned[
        som_prob_columns
    ].to_numpy(dtype=np.float64)

def som_log_blend(p8, p27, psom, weights):
    w8, w27, wsom = weights
    scores = (
        w8 * np.log(np.clip(p8, SOM_EPS, 1.0))
        + w27 * np.log(np.clip(p27, SOM_EPS, 1.0))
        + wsom * np.log(np.clip(psom, SOM_EPS, 1.0))
    )
    return scores.argmax(axis=1)

# 기준 2모델은 0.01 간격, SoM 3모델은 계산량을 제한하기 위해 0.05 간격입니다.
base_candidates = [
    (None, (round(alpha, 2), round(1.0 - alpha, 2), 0.0))
    for alpha in np.linspace(0.0, 1.0, 101)
]
weight_units = int(round(1.0 / SOM_WEIGHT_STEP))
som_candidates = []
for threshold in SOM_BOX_THRESHOLDS:
    for i in range(weight_units + 1):
        for j in range(weight_units + 1 - i):
            w8 = i * SOM_WEIGHT_STEP
            w27 = j * SOM_WEIGHT_STEP
            wsom = 1.0 - w8 - w27
            som_candidates.append(
                (threshold, (round(w8, 4), round(w27, 4), round(wsom, 4)))
            )

def candidate_prediction_matrix(candidates):
    rows = []
    for threshold, weights in candidates:
        psom = (
            som_p8
            if threshold is None
            else som_probability_by_threshold[threshold]
        )
        rows.append(som_log_blend(som_p8, som_p27, psom, weights))
    return np.stack(rows)

base_prediction_matrix = candidate_prediction_matrix(base_candidates)
som_prediction_matrix = candidate_prediction_matrix(som_candidates)

def candidate_tie_penalty(candidate, target):
    threshold, weights = candidate
    threshold_penalty = (
        0.0 if threshold is None else abs(float(threshold) - 0.225)
    )
    return float(np.square(np.asarray(weights) - np.asarray(target)).sum()) + (
        0.01 * threshold_penalty
    )

def select_candidate(prediction_matrix, candidates, train_indices, target):
    scores = (
        prediction_matrix[:, train_indices]
        == som_gold[train_indices][None, :]
    ).sum(axis=1)
    best_indices = np.flatnonzero(scores == scores.max())
    selected = min(
        best_indices,
        key=lambda index: candidate_tie_penalty(candidates[index], target),
    )
    return int(selected), int(scores[selected])

all_indices = np.arange(len(som_gold))
best_base_index, best_base_correct = select_candidate(
    base_prediction_matrix, base_candidates, all_indices, (0.5, 0.5, 0.0)
)
best_som_index, best_som_correct = select_candidate(
    som_prediction_matrix, som_candidates, all_indices, (0.4, 0.4, 0.2)
)
BEST_SOM_THRESHOLD, BEST_SOM_WEIGHTS = som_candidates[best_som_index]

cv_rows = []
rng = np.random.default_rng(SEED + 1701)
for repeat in range(SOM_CV_REPEATS):
    folds = np.array_split(rng.permutation(all_indices), 5)
    for fold_index, validation_indices in enumerate(folds):
        train_indices = np.setdiff1d(
            all_indices, validation_indices, assume_unique=False
        )
        selected_base, _ = select_candidate(
            base_prediction_matrix,
            base_candidates,
            train_indices,
            (0.5, 0.5, 0.0),
        )
        selected_som, _ = select_candidate(
            som_prediction_matrix,
            som_candidates,
            train_indices,
            (0.4, 0.4, 0.2),
        )
        cv_rows.append({
            "repeat": repeat,
            "fold": fold_index,
            "n_valid": len(validation_indices),
            "base_correct": int((
                base_prediction_matrix[selected_base, validation_indices]
                == som_gold[validation_indices]
            ).sum()),
            "som_correct": int((
                som_prediction_matrix[selected_som, validation_indices]
                == som_gold[validation_indices]
            ).sum()),
            "selected_base_alpha_8b": base_candidates[selected_base][1][0],
            "selected_som_threshold": som_candidates[selected_som][0],
            "selected_som_w8": som_candidates[selected_som][1][0],
            "selected_som_w27": som_candidates[selected_som][1][1],
            "selected_som_weight": som_candidates[selected_som][1][2],
        })

som_cv_folds = pd.DataFrame(cv_rows)
som_cv_folds.to_csv(SOM_CV_PATH, index=False)
som_cv_repeat = som_cv_folds.groupby("repeat")[
    ["base_correct", "som_correct"]
].sum()
base_cv_mean = float(som_cv_repeat["base_correct"].mean())
som_cv_mean = float(som_cv_repeat["som_correct"].mean())
som_cv_gain = som_cv_mean - base_cv_mean

best_som_predictions = som_prediction_matrix[best_som_index]
best_som_dataframe = som_pair_base[["id", "answer"]].copy()
best_som_dataframe["pred"] = np.asarray(LETTERS)[best_som_predictions]
best_som_dataframe["correct"] = (
    best_som_predictions == som_gold
)
best_som_dataframe.to_csv(
    SOM_ROOT / "som_best_valid_predictions.csv", index=False
)

best_som_raw = som_valid_by_threshold[BEST_SOM_THRESHOLD]
som_oracle_df = som_pair_base[["id", "answer"]].merge(
    best_som_raw[["id", "pred"]],
    on="id",
    validate="one_to_one",
)
eightb_pred_map_valid = dict(zip(
    eightb_valid["id"], eightb_valid["pred"]
))
adapter_pred_map_valid = dict(zip(
    qwen38_valid_count["id"], qwen38_valid_count["pred"]
))
som_triple_oracle = sum(
    (
        eightb_pred_map_valid[row.id] == row.answer
        or adapter_pred_map_valid[row.id] == row.answer
        or row.pred == row.answer
    )
    for row in som_oracle_df.itertuples()
)

noncount_correct_for_som = int(
    eightb_valid.loc[
        eightb_valid["category"] != "counting", "correct"
    ].sum()
)
projected_som_correct = noncount_correct_for_som + best_som_correct

SOM_GATE_PASSED = (
    best_som_correct >= SOM_MIN_GLOBAL_CORRECT
    and som_cv_gain >= SOM_MIN_CV_GAIN
)

print("\n=== Targeted SoM validation 결정 ===")
print(
    "기존 2모델 global:",
    f"{best_base_correct}/180",
    "weights=", base_candidates[best_base_index][1],
)
print(
    "SoM global:",
    f"{best_som_correct}/180",
    f"threshold={BEST_SOM_THRESHOLD:.2f}",
    "weights(8B,27B,SoM)=", BEST_SOM_WEIGHTS,
)
print(f"3모델 raw oracle: {som_triple_oracle}/180")
print(
    f"반복 5-fold 평균: 기존={base_cv_mean:.2f}/180, "
    f"SoM={som_cv_mean:.2f}/180, gain={som_cv_gain:+.2f}"
)
print(
    f"Projected 전체: {projected_som_correct}/508 "
    f"= {projected_som_correct / 508:.4f}"
)
print(
    "test gate:",
    "PASS" if SOM_GATE_PASSED else "SKIP",
    f"(global>={SOM_MIN_GLOBAL_CORRECT}, cv_gain>={SOM_MIN_CV_GAIN})",
)


In [ ]:
# ===== validation을 통과한 경우에만 test SoM 및 새 제출 생성 =====
run_som_test = bool(SOM_ENABLED and (SOM_GATE_PASSED or FORCE_SOM_TEST))

if run_som_test:
    test_counting_df = test_df[
        test_df["category"] == "counting"
    ].copy().reset_index(drop=True)

    som_test_detections = run_som_detector(
        test_counting_df, "test", SOM_TEST_IMAGE_ROOT
    )
    som_test_detections.to_csv(SOM_TEST_DETECTION_PATH, index=False)

    best_key = som_threshold_key(BEST_SOM_THRESHOLD)
    active_som_test = test_counting_df.merge(
        som_test_detections[
            ["id", "detector_prompt", f"som_path_{best_key}"]
        ],
        on="id",
        how="inner",
        validate="one_to_one",
    )
    active_som_test["som_image_path"] = active_som_test[
        f"som_path_{best_key}"
    ]
    assert len(active_som_test) == len(test_counting_df)

    som8_test = score_dataframe_som(
        active_som_test,
        batch_size=SOM_8B_BATCH_SIZE,
        desc=f"SoM 8B test t={BEST_SOM_THRESHOLD:.2f}",
    )
    som8_test.to_csv(SOM_ROOT / "som8_test.csv", index=False)

    paired_som_test = eightb_test[
        eightb_test["category"] == "counting"
    ][["id", *som_prob_columns]].merge(
        qwen38_test_count[["id", *som_prob_columns]],
        on="id",
        how="inner",
        validate="one_to_one",
        suffixes=("_8b", "_27b"),
    ).merge(
        som8_test[["id", *som_prob_columns]],
        on="id",
        how="inner",
        validate="one_to_one",
    )
    assert len(paired_som_test) == len(test_counting_df)

    test_p8 = paired_som_test[
        [f"prob_{letter}_8b" for letter in LETTERS]
    ].to_numpy(dtype=np.float64)
    test_p27 = paired_som_test[
        [f"prob_{letter}_27b" for letter in LETTERS]
    ].to_numpy(dtype=np.float64)
    test_psom = paired_som_test[
        som_prob_columns
    ].to_numpy(dtype=np.float64)

    som_test_prediction_indices = som_log_blend(
        test_p8, test_p27, test_psom, BEST_SOM_WEIGHTS
    )
    paired_som_test["som_ensemble_pred"] = np.asarray(LETTERS)[
        som_test_prediction_indices
    ]
    paired_som_test.to_csv(
        SOM_ROOT / "paired_counting_test_predictions.csv", index=False
    )
    som_counting_map = dict(zip(
        paired_som_test["id"], paired_som_test["som_ensemble_pred"]
    ))
    eightb_all_test_map = dict(zip(eightb_test["id"], eightb_test["pred"]))

    som_final_answers = [
        som_counting_map[row.id]
        if row.category == "counting"
        else eightb_all_test_map[row.id]
        for row in test_df.itertuples()
    ]
    som_submission = pd.DataFrame({
        "id": test_df["id"],
        "answer": som_final_answers,
    })
    assert len(som_submission) == len(test_df)
    assert som_submission["id"].tolist() == test_df["id"].tolist()
    assert som_submission["answer"].isin(LETTERS).all()
    assert not som_submission["answer"].isna().any()
    som_submission.to_csv(SOM_SUBMISSION_PATH, index=False)

    print("\nSoM 최종 submission:", SOM_SUBMISSION_PATH)
    print(som_submission["answer"].value_counts().sort_index())

    if (
        globals().get("EXPORT_TO_DRIVE", False)
        and globals().get("DRIVE_OUTPUT_DIR") is not None
        and Path(DRIVE_OUTPUT_DIR).exists()
    ):
        som_artifacts = [
            SOM_VALID_DETECTION_PATH,
            SOM_TEST_DETECTION_PATH,
            SOM_CV_PATH,
            SOM_ROOT / "som_best_valid_predictions.csv",
            SOM_ROOT / "som8_test.csv",
            SOM_ROOT / "paired_counting_test_predictions.csv",
            SOM_SUBMISSION_PATH,
        ]
        for artifact in som_artifacts:
            if Path(artifact).exists():
                target = Path(DRIVE_OUTPUT_DIR) / Path(artifact).name
                shutil.copy2(artifact, target)
                print("copied:", target)
else:
    print(
        "SoM이 validation 안전 기준을 통과하지 못해 test 추론을 건너뜁니다. "
        "결과를 확인한 뒤 강제로 시험하려면 FORCE_SOM_TEST=True로 바꾸세요."
    )

# detector는 이후 필요하지 않으므로 항상 해제합니다.
for detector_name in ["som_detector_model", "som_detector_processor"]:
    if detector_name in globals():
        del globals()[detector_name]
gc.collect()
torch.cuda.empty_cache()
print("SoM 종료 메모리:", cuda_memory())
